In [ ]:
############################找高线性相关

In [ ]:
import pandas as pd
import glob
import os

In [ ]:
# 获取当前目录及子目录下所有.xlsx文件
xlsx_files = glob.glob("/p1/data/jtian/SA/input/01_FindLinearCorrelation/*.xlsx")

for xlsx_file in xlsx_files:
    try:
        # 读取Excel文件
        df = pd.read_excel(xlsx_file)
        
        # 生成对应的csv文件名（保持目录结构）
        csv_file = os.path.splitext(xlsx_file)[0] + ".csv"
        
        # 确保目标目录存在
        os.makedirs(os.path.dirname(csv_file), exist_ok=True)
        
        # 转换为CSV文件
        df.to_csv(csv_file, index=False, encoding='utf-8')
        
        print(f"已转换: {xlsx_file} -> {csv_file}")
    except Exception as e:
        print(f"转换失败 {xlsx_file}: {e}")

print(f"转换完成！共转换了 {len(xlsx_files)} 个文件")

In [ ]:
#拼接NN

# 获取所有CSV文件路径
csv_files = glob.glob("/p1/data/jtian/SA/input/01_FindLinearCorrelation/*_NN*.csv")  # 获取当前目录下所有CSV文件

# 读取并拼接所有CSV文件
dataframes = []
for file in csv_files:
    df = pd.read_csv(file)
    dataframes.append(df)

# 上下拼接所有DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)
combined_df = combined_df.sort_values(by='mz', ascending=True)
combined_df.columns = ['mz', '0', '60', '45', '30', '15']
desired_order = ['mz', '0', '15', '30', '45', '60']
combined_df = combined_df[desired_order]
output_filename = "/p1/data/jtian/SA/output/Average intensity_NN.csv"
combined_df.to_csv(output_filename, index=False)

print(f"文件已保存为: {output_filename}")
print(f"总行数: {len(combined_df)}")

print(combined_df.head(10))

In [ ]:
#拼接RMS

# 获取所有CSV文件路径
csv_files = glob.glob("/p1/data/jtian/SA/input/01_FindLinearCorrelation/*_RMS*.csv")  # 获取当前目录下所有CSV文件

# 读取并拼接所有CSV文件
dataframes = []
for file in csv_files:
    df = pd.read_csv(file)
    dataframes.append(df)

# 上下拼接所有DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)
combined_df = combined_df.sort_values(by='mz', ascending=True)
combined_df.columns = ['mz', '0', '60', '45', '30', '15']
desired_order = ['mz', '0', '15', '30', '45', '60']
combined_df = combined_df[desired_order]
output_filename = "/p1/data/jtian/SA/output/Average intensity_RMS.csv"
combined_df.to_csv(output_filename, index=False)

print(f"文件已保存为: {output_filename}")
print(f"总行数: {len(combined_df)}")

print(combined_df.head(10))

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [6]:
df_NN = pd.read_csv("/p1/data/jtian/SA/output/Average intensity_NN.csv")
df_RMS=pd.read_csv("/p1/data/jtian/SA/output/Average intensity_RMS.csv")
print(df_NN.head(10))
print(df_RMS.head(10))

        mz          0          15          30          45          60
0  57.0346   8.641784   33.473495   47.207394   58.450836   60.037590
1  58.0302   5.487052   30.011187   45.398209   55.877441   58.912498
2  59.0139  26.446842  279.444824  422.921326  488.085358  524.221436
3  61.9884  91.392006   48.898109   45.010029   46.626850   54.080418
4  62.9643  11.605034  324.540833  414.852875  426.807343  388.374573
5  64.0192  65.862640   80.500839   74.225220   62.215355   56.183300
6  67.0191  11.261073   61.642776   85.846786  103.593834  114.299637
7  67.9670  53.651134   66.132904   69.507217   67.208244   57.065540
8  68.0143  21.494078   55.650810   65.008369   66.403023   65.100678
9  68.9977  34.493492   90.518829  122.786736  142.599747  152.527939
        mz          0         15         30          45          60
0  57.0346   1.086315   5.902466  10.106243   14.920744   16.912420
1  58.0302   0.686013   5.286836   9.705569   14.354864   16.709291
2  59.0139   3.320908  49.

In [11]:
#使用sklearn计算R²
def calculate_r2_sklearn(df, x_values=[0, 15, 30, 45, 60]):
    # 将x_values转换为二维数组
    X = np.array(x_values).reshape(-1, 1)
    r2_scores = []
    
    for idx, row in df.iterrows():
        y_values = row[1:].values
        
        # 创建线性回归模型
        model = LinearRegression()
        model.fit(X, y_values)
        
        # 预测并计算R²
        y_pred = model.predict(X)
        r2 = r2_score(y_values, y_pred)
        r2_scores.append(r2)
    
    return r2_scores



In [12]:
r2_values_NN = calculate_r2_sklearn(df_NN)
df_NN['R2'] = r2_values_NN

In [13]:
r2_values_RMS = calculate_r2_sklearn(df_RMS)
df_RMS['R2'] = r2_values_RMS

In [14]:
print(df_NN.head(10))
print(df_RMS.head(10))

        mz          0          15          30          45          60  \
0  57.0346   8.641784   33.473495   47.207394   58.450836   60.037590   
1  58.0302   5.487052   30.011187   45.398209   55.877441   58.912498   
2  59.0139  26.446842  279.444824  422.921326  488.085358  524.221436   
3  61.9884  91.392006   48.898109   45.010029   46.626850   54.080418   
4  62.9643  11.605034  324.540833  414.852875  426.807343  388.374573   
5  64.0192  65.862640   80.500839   74.225220   62.215355   56.183300   
6  67.0191  11.261073   61.642776   85.846786  103.593834  114.299637   
7  67.9670  53.651134   66.132904   69.507217   67.208244   57.065540   
8  68.0143  21.494078   55.650810   65.008369   66.403023   65.100678   
9  68.9977  34.493492   90.518829  122.786736  142.599747  152.527939   

         R2  
0  0.903118  
1  0.914475  
2  0.882109  
3  0.392058  
4  0.610441  
5  0.380441  
6  0.915412  
7  0.032527  
8  0.659635  
9  0.911859  
        mz          0         15         3

In [15]:
print(len(df_NN[df_NN.R2>0.9]))
print(len(df_NN[df_NN.R2>0.95]))
print(len(df_RMS[df_RMS.R2>0.9]))
print(len(df_RMS[df_RMS.R2>0.95]))

914
514
2325
1625


In [18]:
mz09=df_RMS[df_RMS.R2>0.9][['mz','R2']]
mz09.to_csv('/p1/data/jtian/SA/output/mz09.csv', index=False)